# Real-Time Remote Patient Monitoring - Synthetic Data Generator

## Overview
This notebook generates synthetic patient vital signs data for demonstrating the Real-Time Remote Patient Monitoring use case with Microsoft Fabric Real-Time Intelligence.

## What it Simulates
- **Patient demographics** with chronic conditions (heart failure, COPD, diabetes)
- **Device registry** mapping patients to monitoring devices
- **Vital signs streaming data** (heart rate, SpO2, blood pressure, glucose, weight)
- **Anomalous patterns** representing clinical deterioration scenarios

## Tables/Events Created
| Table | Description | Volume |
|-------|-------------|--------|
| `patients` | Patient demographics and conditions | 100-1000 patients |
| `devices` | Device registry with patient assignments | 1-3 devices per patient |
| `care_teams` | Care team member assignments | 5-10 providers per patient |
| `vital_signs` | Streaming vital sign readings | 50-100 readings/patient/day |
| `alerts` | Generated clinical alerts | Based on threshold violations |

## How to Tune Volume
Adjust the parameters in the Configuration cell below.

## Known Limitations
- Synthetic data only - not based on real clinical patterns
- Simplified anomaly patterns for demonstration purposes
- Does not simulate all device types or clinical conditions

## Setup

### How to Run in Microsoft Fabric
1. Import this notebook into your Microsoft Fabric workspace
2. Attach a Lakehouse to this notebook
3. Run the cells in order
4. The generated data will be written to Delta tables in your Lakehouse

### For Streaming Simulation
- Schedule this notebook to run every 1-5 minutes
- Set `STREAMING_MODE = True` in the configuration
- Each run will generate new readings appended to the `vital_signs` table

### Idempotent Behavior
- `OVERWRITE_MODE = True`: Recreates all tables (for initial setup)
- `OVERWRITE_MODE = False`: Appends new readings (for streaming simulation)

In [ ]:
# Install required packages (run once)
!pip install faker

In [ ]:
# Configuration Parameters

# Scale factor (adjust based on your needs)
NUM_PATIENTS = 100              # Number of patients to generate
READINGS_PER_PATIENT = 50       # Vital sign readings per patient (for batch mode)
ANOMALY_RATE = 0.05             # 5% of readings will be anomalous

# Date range for historical data
from datetime import datetime, timedelta
START_DATE = datetime.now() - timedelta(days=7)
END_DATE = datetime.now()

# Streaming mode settings
STREAMING_MODE = False          # Set True for incremental streaming simulation
STREAMING_BATCH_SIZE = 10       # Readings per patient per streaming run

# Write mode
OVERWRITE_MODE = True           # True: recreate tables, False: append

# Random seed for reproducibility
RANDOM_SEED = 42

# Lakehouse paths (update if needed)
LAKEHOUSE_PATH = "Tables"       # Default Fabric Lakehouse tables path

print(f"Configuration loaded:")
print(f"  - Patients: {NUM_PATIENTS}")
print(f"  - Readings per patient: {READINGS_PER_PATIENT}")
print(f"  - Date range: {START_DATE.date()} to {END_DATE.date()}")
print(f"  - Streaming mode: {STREAMING_MODE}")
print(f"  - Overwrite mode: {OVERWRITE_MODE}")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import uuid

# Set random seeds for reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
fake = Faker()
Faker.seed(RANDOM_SEED)

print("Libraries imported successfully")

## Data Contract

### Patients Table Schema
| Column | Type | Description |
|--------|------|-------------|
| patient_id | string | Unique patient identifier |
| first_name | string | Patient first name |
| last_name | string | Patient last name |
| date_of_birth | date | Date of birth |
| gender | string | M/F |
| primary_condition | string | Primary chronic condition |
| secondary_conditions | string | Comma-separated list |
| enrollment_date | date | RPM program enrollment date |
| risk_tier | string | Low/Medium/High |

### Vital Signs Table Schema
| Column | Type | Description |
|--------|------|-------------|
| reading_id | string | Unique reading identifier |
| patient_id | string | Foreign key to patients |
| device_id | string | Foreign key to devices |
| timestamp | timestamp | Reading timestamp |
| vital_type | string | heart_rate/spo2/blood_pressure/glucose/weight |
| value | float | Primary measurement value |
| value_secondary | float | Secondary value (e.g., diastolic BP) |
| unit | string | Measurement unit |
| is_anomaly | boolean | Whether this is an anomalous reading |

In [ ]:
# Define clinical reference data

CONDITIONS = [
    "Heart Failure",
    "COPD",
    "Type 2 Diabetes",
    "Hypertension",
    "Chronic Kidney Disease",
    "Atrial Fibrillation"
]

SECONDARY_CONDITIONS = [
    "Obesity",
    "Sleep Apnea",
    "Depression",
    "Anxiety",
    "Arthritis",
    "Hypothyroidism"
]

DEVICE_TYPES = [
    {"type": "heart_rate_monitor", "manufacturer": "CardioTech", "model": "CT-100"},
    {"type": "pulse_oximeter", "manufacturer": "OxyWatch", "model": "OW-Pro"},
    {"type": "blood_pressure_cuff", "manufacturer": "BPMaster", "model": "BPM-200"},
    {"type": "glucose_monitor", "manufacturer": "GlucoSense", "model": "GS-Continuous"},
    {"type": "smart_scale", "manufacturer": "WeightWise", "model": "WW-500"}
]

CARE_TEAM_ROLES = [
    "Primary Care Physician",
    "Cardiologist",
    "Pulmonologist",
    "Endocrinologist",
    "Care Coordinator",
    "Registered Nurse",
    "Pharmacist"
]

# Normal vital sign ranges (by condition)
VITAL_RANGES = {
    "heart_rate": {"normal_min": 60, "normal_max": 100, "unit": "bpm"},
    "spo2": {"normal_min": 95, "normal_max": 100, "unit": "%"},
    "blood_pressure_systolic": {"normal_min": 90, "normal_max": 140, "unit": "mmHg"},
    "blood_pressure_diastolic": {"normal_min": 60, "normal_max": 90, "unit": "mmHg"},
    "glucose": {"normal_min": 70, "normal_max": 180, "unit": "mg/dL"},
    "weight": {"normal_min": 120, "normal_max": 250, "unit": "lbs"}
}

print("Reference data defined")

In [ ]:
# Generate Patients

def generate_patients(n):
    patients = []
    for _ in range(n):
        # Generate age-appropriate for chronic conditions (50-85 years)
        age = random.randint(50, 85)
        dob = datetime.now() - timedelta(days=age*365 + random.randint(0, 364))
        
        # Assign primary condition
        primary_condition = random.choice(CONDITIONS)
        
        # 60% chance of secondary conditions
        secondary = []
        if random.random() < 0.6:
            num_secondary = random.randint(1, 3)
            secondary = random.sample(SECONDARY_CONDITIONS, num_secondary)
        
        # Risk tier based on conditions
        risk_tier = "Low"
        if primary_condition in ["Heart Failure", "COPD"] or len(secondary) >= 2:
            risk_tier = "High"
        elif len(secondary) >= 1:
            risk_tier = "Medium"
        
        patient = {
            "patient_id": str(uuid.uuid4()),
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "date_of_birth": dob.date(),
            "gender": random.choice(["M", "F"]),
            "primary_condition": primary_condition,
            "secondary_conditions": ", ".join(secondary) if secondary else None,
            "enrollment_date": (datetime.now() - timedelta(days=random.randint(30, 365))).date(),
            "risk_tier": risk_tier,
            "baseline_heart_rate": random.randint(65, 85),
            "baseline_weight": random.randint(140, 220)
        }
        patients.append(patient)
    
    return pd.DataFrame(patients)

# Generate patients
df_patients = generate_patients(NUM_PATIENTS)
print(f"Generated {len(df_patients)} patients")
df_patients.head()

In [ ]:
# Generate Devices

def generate_devices(patients_df):
    devices = []
    
    for _, patient in patients_df.iterrows():
        # Each patient gets 2-4 devices based on conditions
        patient_devices = []
        
        # All patients get heart rate monitor
        patient_devices.append(DEVICE_TYPES[0])  # heart_rate_monitor
        
        # Condition-specific devices
        if patient["primary_condition"] in ["Heart Failure", "COPD"]:
            patient_devices.append(DEVICE_TYPES[1])  # pulse_oximeter
            patient_devices.append(DEVICE_TYPES[4])  # smart_scale
        
        if patient["primary_condition"] == "Hypertension" or "Hypertension" in str(patient["secondary_conditions"]):
            patient_devices.append(DEVICE_TYPES[2])  # blood_pressure_cuff
        
        if patient["primary_condition"] == "Type 2 Diabetes":
            patient_devices.append(DEVICE_TYPES[3])  # glucose_monitor
        
        # Create device records
        for device_type in patient_devices:
            device = {
                "device_id": str(uuid.uuid4()),
                "patient_id": patient["patient_id"],
                "device_type": device_type["type"],
                "manufacturer": device_type["manufacturer"],
                "model": device_type["model"],
                "serial_number": fake.bothify(text="??######").upper(),
                "activation_date": patient["enrollment_date"],
                "battery_level": random.randint(20, 100),
                "last_sync": datetime.now() - timedelta(minutes=random.randint(1, 60)),
                "status": random.choices(["active", "low_battery", "offline"], weights=[0.9, 0.07, 0.03])[0]
            }
            devices.append(device)
    
    return pd.DataFrame(devices)

# Generate devices
df_devices = generate_devices(df_patients)
print(f"Generated {len(df_devices)} devices for {NUM_PATIENTS} patients")
df_devices.head()

In [ ]:
# Generate Care Teams

def generate_care_teams(patients_df):
    care_teams = []
    
    # Pre-generate some provider names
    providers = {role: [(fake.name(), fake.email()) for _ in range(10)] for role in CARE_TEAM_ROLES}
    
    for _, patient in patients_df.iterrows():
        # Assign care team members
        assigned_roles = ["Primary Care Physician", "Care Coordinator", "Registered Nurse"]
        
        # Add specialists based on condition
        if patient["primary_condition"] in ["Heart Failure", "Atrial Fibrillation"]:
            assigned_roles.append("Cardiologist")
        if patient["primary_condition"] == "COPD":
            assigned_roles.append("Pulmonologist")
        if patient["primary_condition"] == "Type 2 Diabetes":
            assigned_roles.append("Endocrinologist")
        
        for role in assigned_roles:
            provider_name, provider_email = random.choice(providers[role])
            care_team = {
                "assignment_id": str(uuid.uuid4()),
                "patient_id": patient["patient_id"],
                "provider_name": provider_name,
                "provider_email": provider_email,
                "role": role,
                "is_primary_contact": role == "Care Coordinator",
                "assignment_date": patient["enrollment_date"]
            }
            care_teams.append(care_team)
    
    return pd.DataFrame(care_teams)

# Generate care teams
df_care_teams = generate_care_teams(df_patients)
print(f"Generated {len(df_care_teams)} care team assignments")
df_care_teams.head()

In [ ]:
# Generate Vital Signs Readings

def generate_vital_reading(patient, device, timestamp, force_anomaly=False):
    """Generate a single vital sign reading"""
    device_type = device["device_type"]
    is_anomaly = force_anomaly or random.random() < ANOMALY_RATE
    
    reading = {
        "reading_id": str(uuid.uuid4()),
        "patient_id": patient["patient_id"],
        "device_id": device["device_id"],
        "timestamp": timestamp,
        "is_anomaly": is_anomaly
    }
    
    if device_type == "heart_rate_monitor":
        baseline = patient["baseline_heart_rate"]
        if is_anomaly:
            # Tachycardia or bradycardia
            value = random.choice([random.randint(40, 55), random.randint(110, 150)])
        else:
            value = baseline + random.randint(-10, 15)
        reading.update({
            "vital_type": "heart_rate",
            "value": value,
            "value_secondary": None,
            "unit": "bpm"
        })
    
    elif device_type == "pulse_oximeter":
        if is_anomaly:
            value = random.randint(85, 92)  # Hypoxemia
        else:
            value = random.randint(95, 100)
        reading.update({
            "vital_type": "spo2",
            "value": value,
            "value_secondary": None,
            "unit": "%"
        })
    
    elif device_type == "blood_pressure_cuff":
        if is_anomaly:
            # Hypertensive crisis or hypotension
            systolic = random.choice([random.randint(80, 90), random.randint(160, 200)])
            diastolic = random.choice([random.randint(50, 58), random.randint(100, 120)])
        else:
            systolic = random.randint(110, 135)
            diastolic = random.randint(70, 85)
        reading.update({
            "vital_type": "blood_pressure",
            "value": systolic,
            "value_secondary": diastolic,
            "unit": "mmHg"
        })
    
    elif device_type == "glucose_monitor":
        if is_anomaly:
            # Hypoglycemia or hyperglycemia
            value = random.choice([random.randint(45, 65), random.randint(250, 400)])
        else:
            value = random.randint(80, 160)
        reading.update({
            "vital_type": "glucose",
            "value": value,
            "value_secondary": None,
            "unit": "mg/dL"
        })
    
    elif device_type == "smart_scale":
        baseline = patient["baseline_weight"]
        if is_anomaly:
            # Sudden weight gain (fluid retention in HF)
            value = baseline + random.randint(3, 8)
        else:
            value = baseline + random.uniform(-1.5, 1.5)
        reading.update({
            "vital_type": "weight",
            "value": round(value, 1),
            "value_secondary": None,
            "unit": "lbs"
        })
    
    return reading

def generate_vital_signs(patients_df, devices_df, start_date, end_date, readings_per_patient):
    """Generate vital signs for all patients"""
    all_readings = []
    
    for _, patient in patients_df.iterrows():
        patient_devices = devices_df[devices_df["patient_id"] == patient["patient_id"]]
        
        for _, device in patient_devices.iterrows():
            # Generate readings for this device
            for i in range(readings_per_patient):
                # Random timestamp within date range
                time_delta = end_date - start_date
                random_seconds = random.randint(0, int(time_delta.total_seconds()))
                timestamp = start_date + timedelta(seconds=random_seconds)
                
                reading = generate_vital_reading(patient, device, timestamp)
                all_readings.append(reading)
    
    return pd.DataFrame(all_readings)

# Generate vital signs
if STREAMING_MODE:
    # Streaming mode: generate just recent readings
    stream_start = datetime.now() - timedelta(minutes=5)
    stream_end = datetime.now()
    df_vital_signs = generate_vital_signs(df_patients, df_devices, stream_start, stream_end, STREAMING_BATCH_SIZE)
else:
    # Batch mode: generate historical backfill
    df_vital_signs = generate_vital_signs(df_patients, df_devices, START_DATE, END_DATE, READINGS_PER_PATIENT)

# Sort by timestamp
df_vital_signs = df_vital_signs.sort_values("timestamp")

print(f"Generated {len(df_vital_signs)} vital sign readings")
print(f"Anomaly rate: {df_vital_signs['is_anomaly'].mean():.2%}")
df_vital_signs.head(10)

In [ ]:
# Generate Alerts from Anomalous Readings

def generate_alerts(vital_signs_df, patients_df):
    """Generate clinical alerts based on anomalous readings"""
    anomalies = vital_signs_df[vital_signs_df["is_anomaly"] == True].copy()
    alerts = []
    
    for _, reading in anomalies.iterrows():
        patient = patients_df[patients_df["patient_id"] == reading["patient_id"]].iloc[0]
        
        # Determine alert severity and message
        severity = "Warning"
        message = ""
        
        if reading["vital_type"] == "heart_rate":
            if reading["value"] > 120:
                severity = "Critical"
                message = f"Tachycardia detected: HR {reading['value']} bpm"
            else:
                message = f"Bradycardia detected: HR {reading['value']} bpm"
        
        elif reading["vital_type"] == "spo2":
            if reading["value"] < 90:
                severity = "Critical"
                message = f"Severe hypoxemia: SpO2 {reading['value']}%"
            else:
                message = f"Low oxygen saturation: SpO2 {reading['value']}%"
        
        elif reading["vital_type"] == "blood_pressure":
            if reading["value"] > 180 or reading["value"] < 90:
                severity = "Critical"
                message = f"Blood pressure alert: {reading['value']}/{reading['value_secondary']} mmHg"
            else:
                message = f"Abnormal blood pressure: {reading['value']}/{reading['value_secondary']} mmHg"
        
        elif reading["vital_type"] == "glucose":
            if reading["value"] < 60 or reading["value"] > 300:
                severity = "Critical"
                message = f"Glucose emergency: {reading['value']} mg/dL"
            else:
                message = f"Abnormal glucose: {reading['value']} mg/dL"
        
        elif reading["vital_type"] == "weight":
            weight_change = reading["value"] - patient["baseline_weight"]
            if weight_change > 5:
                severity = "Critical"
            message = f"Weight gain alert: +{weight_change:.1f} lbs from baseline"
        
        alert = {
            "alert_id": str(uuid.uuid4()),
            "patient_id": reading["patient_id"],
            "reading_id": reading["reading_id"],
            "alert_timestamp": reading["timestamp"] + timedelta(seconds=random.randint(1, 30)),
            "vital_type": reading["vital_type"],
            "severity": severity,
            "message": message,
            "status": random.choices(["New", "Acknowledged", "Resolved"], weights=[0.4, 0.35, 0.25])[0],
            "acknowledged_by": fake.name() if random.random() > 0.4 else None,
            "acknowledged_at": None  # Would be populated when acknowledged
        }
        alerts.append(alert)
    
    return pd.DataFrame(alerts)

# Generate alerts
df_alerts = generate_alerts(df_vital_signs, df_patients)
print(f"Generated {len(df_alerts)} alerts")
print(f"Critical alerts: {len(df_alerts[df_alerts['severity'] == 'Critical'])}")
df_alerts.head()

## Quick Validation

In [ ]:
# Validation Summary

print("="*60)
print("DATA VALIDATION SUMMARY")
print("="*60)

# Row counts
print(f"\n📊 Row Counts:")
print(f"   Patients:     {len(df_patients):,}")
print(f"   Devices:      {len(df_devices):,}")
print(f"   Care Teams:   {len(df_care_teams):,}")
print(f"   Vital Signs:  {len(df_vital_signs):,}")
print(f"   Alerts:       {len(df_alerts):,}")

# Null checks
print(f"\n🔍 Null Check (critical columns):")
print(f"   Patients with null patient_id: {df_patients['patient_id'].isnull().sum()}")
print(f"   Vital signs with null reading_id: {df_vital_signs['reading_id'].isnull().sum()}")
print(f"   Vital signs with null value: {df_vital_signs['value'].isnull().sum()}")

# Key integrity
print(f"\n🔗 Referential Integrity:")
vital_patient_ids = set(df_vital_signs['patient_id'].unique())
patient_ids = set(df_patients['patient_id'].unique())
orphan_readings = vital_patient_ids - patient_ids
print(f"   Orphan vital sign readings: {len(orphan_readings)}")

# Distribution checks
print(f"\n📈 Distributions:")
print(f"   Risk tier distribution:")
for tier, count in df_patients['risk_tier'].value_counts().items():
    print(f"      {tier}: {count} ({count/len(df_patients):.1%})")

print(f"\n   Vital sign types:")
for vtype, count in df_vital_signs['vital_type'].value_counts().items():
    print(f"      {vtype}: {count:,}")

print(f"\n   Alert severity distribution:")
for severity, count in df_alerts['severity'].value_counts().items():
    print(f"      {severity}: {count} ({count/len(df_alerts):.1%})")

print("\n" + "="*60)
print("✅ Validation complete")
print("="*60)

In [ ]:
# Sanity Visualizations (if matplotlib available)

try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Heart rate distribution
    hr_readings = df_vital_signs[df_vital_signs['vital_type'] == 'heart_rate']
    axes[0, 0].hist(hr_readings['value'], bins=30, edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(x=60, color='green', linestyle='--', label='Normal range')
    axes[0, 0].axvline(x=100, color='green', linestyle='--')
    axes[0, 0].set_title('Heart Rate Distribution')
    axes[0, 0].set_xlabel('BPM')
    axes[0, 0].legend()
    
    # SpO2 distribution
    spo2_readings = df_vital_signs[df_vital_signs['vital_type'] == 'spo2']
    if len(spo2_readings) > 0:
        axes[0, 1].hist(spo2_readings['value'], bins=20, edgecolor='black', alpha=0.7, color='orange')
        axes[0, 1].axvline(x=95, color='green', linestyle='--', label='Normal threshold')
        axes[0, 1].set_title('SpO2 Distribution')
        axes[0, 1].set_xlabel('%')
        axes[0, 1].legend()
    
    # Readings over time
    readings_by_hour = df_vital_signs.copy()
    readings_by_hour['hour'] = pd.to_datetime(readings_by_hour['timestamp']).dt.hour
    hourly_counts = readings_by_hour.groupby('hour').size()
    axes[1, 0].bar(hourly_counts.index, hourly_counts.values, color='steelblue')
    axes[1, 0].set_title('Readings by Hour of Day')
    axes[1, 0].set_xlabel('Hour')
    axes[1, 0].set_ylabel('Count')
    
    # Alert severity pie chart
    severity_counts = df_alerts['severity'].value_counts()
    colors = {'Critical': 'red', 'Warning': 'orange'}
    axes[1, 1].pie(severity_counts.values, labels=severity_counts.index, autopct='%1.1f%%',
                   colors=[colors.get(s, 'gray') for s in severity_counts.index])
    axes[1, 1].set_title('Alert Severity Distribution')
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("matplotlib not available - skipping visualizations")

## Write Data to Lakehouse

In [ ]:
# Write to Delta tables in Lakehouse
# Note: This cell is designed for Microsoft Fabric. 
# Uncomment the spark commands when running in Fabric.

write_mode = "overwrite" if OVERWRITE_MODE else "append"

print(f"Writing data to Lakehouse (mode: {write_mode})...")

# Convert pandas DataFrames to Spark DataFrames and write
# Uncomment these lines when running in Microsoft Fabric:

# spark.createDataFrame(df_patients).write.mode(write_mode).format("delta").saveAsTable("patients")
# print("  ✓ patients table written")

# spark.createDataFrame(df_devices).write.mode(write_mode).format("delta").saveAsTable("devices")
# print("  ✓ devices table written")

# spark.createDataFrame(df_care_teams).write.mode(write_mode).format("delta").saveAsTable("care_teams")
# print("  ✓ care_teams table written")

# spark.createDataFrame(df_vital_signs).write.mode(write_mode).format("delta").saveAsTable("vital_signs")
# print("  ✓ vital_signs table written")

# spark.createDataFrame(df_alerts).write.mode(write_mode).format("delta").saveAsTable("alerts")
# print("  ✓ alerts table written")

# For local testing, save as CSV
df_patients.to_csv("patients.csv", index=False)
df_devices.to_csv("devices.csv", index=False)
df_care_teams.to_csv("care_teams.csv", index=False)
df_vital_signs.to_csv("vital_signs.csv", index=False)
df_alerts.to_csv("alerts.csv", index=False)

print("\n✅ Data written successfully!")
print("\nFor Microsoft Fabric, uncomment the spark.createDataFrame lines above.")

## Sample Queries

Use these KQL queries in Eventhouse to analyze the data:

```kql
// Get all critical alerts in the last 24 hours
alerts
| where alert_timestamp > ago(24h)
| where severity == "Critical"
| join kind=inner patients on patient_id
| project alert_timestamp, first_name, last_name, vital_type, message, severity
| order by alert_timestamp desc

// Heart rate trends for a specific patient
vital_signs
| where vital_type == "heart_rate"
| where patient_id == "<patient_id>"
| summarize avg_hr = avg(value), max_hr = max(value), min_hr = min(value) by bin(timestamp, 1h)
| render timechart

// Patients with highest anomaly rates
vital_signs
| summarize total_readings = count(), anomalies = countif(is_anomaly == true) by patient_id
| extend anomaly_rate = todouble(anomalies) / total_readings
| top 10 by anomaly_rate desc
| join kind=inner patients on patient_id
| project first_name, last_name, primary_condition, risk_tier, anomaly_rate, anomalies, total_readings
```